In [1]:
"""Kaggle T4 x2 / ComfyUI MiniMax-H3：整段粘贴到一个 Python 单元运行。
启用 Settings > Accelerator > GPU T4 x2、Internet。
采用 FL2VA pruned Q4_K_M + H3 专用 Qwen3-VL NVFP4 AWQ，原生 H3 视频/音频节点。
官方 T2V/I2V 模板已内嵌；视频模型使用 GGUF 加载器，文本编码器使用原生 CLIPLoader，并将文本编码器指定到第二张 GPU。
这是跨卡分配组件，不是显存合并。没有实测当前 Kaggle 的生成速度和峰值内存。
"""
import base64
import hashlib
import json
import os
from pathlib import Path
import re
import shutil
import socket
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zlib

# ==================== 配置 ====================
ROOT = Path("/tmp/ComfyUI")
PORT = 8188
LOCAL_URL = f"http://127.0.0.1:{PORT}"
USE_SECOND_GPU = True        # 2 卡：DiT / VAE 默认 GPU0，文本编码器 GPU1
DURATION = 2.0               # 首轮先测试短片；实际帧数按官方 17k+5 网格取整
MEGAPIXELS = 0.1             # 低分辨率试跑，成功后逐步改 0.2 / 0.4；非质量推荐值
TURBO_STEPS = 4
UPDATE_CODE = False           # 默认复用已安装版本；新环境仍自动克隆，需更新时改 True
START_TUNNEL = True
PROMPT = ("Cinematic live-action shot. A small orange cat walks slowly on a rainy street, "
          "neon lights reflecting in puddles. Slow tracking camera, natural motion. "
          "Audio: gentle rain and quiet footsteps. No subtitles or watermarks.")

MANIFEST = [{'repo': 'leejet/MiniMax-H3-GGUF', 'revision': 'd9c4c6312b4728a68a15a35626d84775a6523783', 'file': 'minimax_h3_fl2va_pruned-Q4_K_M.gguf', 'size': 11420663904, 'sha256': 'dd948e08ad0ba3c71bd42f368e283dd82e790f5122a63b276e22a3e0283d0c10', 'folder': 'diffusion_models'}, {'repo': 'Comfy-Org/MiniMax-H3', 'revision': '54472221d27c695fca9621b0c3ed52f06c4af707', 'file': 'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 'size': 15687142551, 'sha256': '35a88d51044231fe332301d7a62aa81e3f2cba62febeb446e2c1e3e0ef76f2c6', 'folder': 'text_encoders'}, {'repo': 'Comfy-Org/MiniMax-H3', 'revision': '54472221d27c695fca9621b0c3ed52f06c4af707', 'file': 'vae/minimax_h3_video_vae_fp16.safetensors', 'size': 5207808496, 'sha256': '7c1f131492e7eddacaac9069a61b81bdd39de5cc96561e677c5eab1cdce5e522', 'folder': 'vae'}, {'repo': 'Comfy-Org/MiniMax-H3', 'revision': '54472221d27c695fca9621b0c3ed52f06c4af707', 'file': 'vae/minimax_h3_audio_vae_fp32.safetensors', 'size': 605254808, 'sha256': '8e505d95dd1561d47abd43d4238fd40d9bb1ae9e147ed0a4cba778d76ae4db48', 'folder': 'vae'}, {'repo': 'Comfy-Org/MiniMax-H3', 'revision': '54472221d27c695fca9621b0c3ed52f06c4af707', 'file': 'loras/minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors', 'size': 1956192992, 'sha256': 'c396a9a06f58399e9df9754b18299818d84a2ddd371724ba48fe4a41221437dc', 'folder': 'loras'}]
TEMPLATE_BUNDLE = '...'


def run(*args, cwd=None):
    subprocess.run(list(map(str, args)), cwd=cwd, check=True)


def tail(path):
    path = Path(path)
    if not path.exists():
        return "尚无日志"
    with path.open("rb") as f:
        f.seek(max(0, path.stat().st_size - 12000))
        return f.read().decode("utf-8", "replace")


def request_json(path, timeout=10):
    # 本地服务不经过 Notebook 的 HTTP_PROXY。
    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
    with opener.open(LOCAL_URL + path, timeout=timeout) as r:
        return json.load(r)


def ready():
    try:
        return "system" in request_json("/system_stats", timeout=2)
    except Exception:
        return False


def stop_owned(name):
    process = globals().get(name)
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=5)



def prepare_python_env(env_dir, base_python):
    """跳过 ensurepip，修复上次失败留下的半成品 venv，不删除已有依赖。"""
    env_dir = Path(env_dir)
    python = env_dir / "bin/python"
    # 每次重写解释器配置；不能只判断 bin/python 是否存在。
    run(base_python, "-m", "venv", "--without-pip", "--system-site-packages", env_dir)
    probe = subprocess.run([str(python), "-c",
        "import sys; print(sys.prefix); assert sys.prefix != sys.base_prefix"],
        capture_output=True, text=True)
    if probe.returncode or Path(probe.stdout.strip()).resolve() != env_dir.resolve():
        raise RuntimeError("虚拟环境解释器检查失败：" + probe.stderr[-3000:])
    pip_probe = subprocess.run([str(python), "-m", "pip", "--version"],
                               capture_output=True, text=True)
    if pip_probe.returncode:
        # pip 官方支持使用外部 pip 管理尚未安装 pip 的虚拟环境。
        manager = subprocess.run([str(base_python), "-m", "pip", "--version"],
                                 capture_output=True, text=True)
        if manager.returncode:
            raise RuntimeError("Notebook 的 Python 也无法调用 pip：" + manager.stderr[-3000:] +
                               "\n请在 Notebook 运行 %pip --version，提供完整输出。")
        result = subprocess.run([str(base_python), "-m", "pip", "--python", str(python),
                                 "install", "pip"], capture_output=True, text=True)
        if result.returncode:
            raise RuntimeError("用外部 pip 初始化虚拟环境失败：\n" +
                               result.stdout[-2000:] + result.stderr[-4000:])
        run(python, "-m", "pip", "--version")
    else:
        print(pip_probe.stdout.strip(), flush=True)
    return python


def clone_or_update(url, folder):
    folder = Path(folder)
    if not folder.exists():
        run("git", "clone", "--depth", "1", url, folder)
    elif not (folder / ".git").is_dir():
        raise RuntimeError(f"{folder} 已存在但不是 git 仓库；请换 ROOT 或先移开该目录。")
    elif UPDATE_CODE:
        changes = subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"], cwd=folder, text=True)
        if changes.strip():
            raise RuntimeError(f"{folder} 有本地代码改动，未覆盖。请保留改动后手工更新，或设置 UPDATE_CODE=False。")
        run("git", "pull", "--ff-only", cwd=folder)


def verify(path, item):
    path = Path(path)
    if not path.is_file() or path.stat().st_size != item["size"]:
        return False
    print("校验 SHA-256：", path.name, flush=True)
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest() == item["sha256"]


def download_model(item):
    target = ROOT / "models" / item["folder"] / Path(item["file"]).name
    target.parent.mkdir(parents=True, exist_ok=True)
    if verify(target, item):
        print("复用：", target.name, flush=True)
        return
    temporary = target.with_name(target.name + ".part")
    url = f"https://huggingface.co/{item['repo']}/resolve/{item['revision']}/{item['file']}"
    for attempt in range(1, 4):
        try:
            offset = temporary.stat().st_size if temporary.exists() else 0
            if offset >= item["size"]:
                if verify(temporary, item):
                    temporary.replace(target)
                    return
                temporary.unlink()
                offset = 0
            headers = {"User-Agent": "Kaggle-ComfyUI-H3/1.0"}
            if offset:
                headers["Range"] = f"bytes={offset}-"
            print(f"下载 {target.name}，已完成 {offset / 2**30:.2f} GiB，尝试 {attempt}/3", flush=True)
            with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=120) as r:
                append = offset > 0 and r.status == 206
                if append and not r.headers.get("Content-Range", "").startswith(f"bytes {offset}-"):
                    raise RuntimeError("服务端断点位置不符")
                if r.status not in (200, 206):
                    raise RuntimeError(f"HTTP {r.status}")
                if r.status == 206 and not append:
                    raise RuntimeError("意外的部分文件响应")
                received = offset if append else 0
                last_print = time.monotonic()
                with temporary.open("ab" if append else "wb") as f:
                    while True:
                        chunk = r.read(8 * 1024 * 1024)
                        if not chunk:
                            break
                        received += len(chunk)
                        if received > item["size"]:
                            raise RuntimeError("下载大小超过预期")
                        f.write(chunk)
                        if time.monotonic() - last_print > 20:
                            print(f"  {received / 2**30:.2f} / {item['size'] / 2**30:.2f} GiB", flush=True)
                            last_print = time.monotonic()
            if not verify(temporary, item):
                if temporary.stat().st_size == item["size"]:
                    temporary.unlink()  # 完整但摘要错误的文件不能续传
                raise RuntimeError("文件长度或 SHA-256 不符")
            temporary.replace(target)
            return
        except Exception as e:
            print(f"下载失败：{type(e).__name__}: {e}", flush=True)
            if attempt == 3:
                raise RuntimeError(f"下载 {target.name} 失败；保留断点，下次重跑可续传。") from e
            time.sleep(2 * attempt)


def iter_graphs(workflow):
    yield workflow
    for graph in workflow.get("definitions", {}).get("subgraphs", []):
        yield graph


def set_widget(node, key, value):
    named = node.get("widgets_values_named", {})
    if key not in named:
        return
    keys = list(named)
    index = keys.index(key)
    if index < len(node.get("widgets_values", [])):
        node["widgets_values"][index] = value
    named[key] = value


def patch_workflow(workflow, second_gpu):
    replacements = {
        "minimax_h3_fl2va_pruned_int8_convrot.safetensors": MANIFEST[0]["file"],
        "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors": Path(MANIFEST[1]["file"]).name,
        "minimax_h3_video_vae_int8_convrot.safetensors": Path(MANIFEST[2]["file"]).name,
        "minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors": Path(MANIFEST[4]["file"]).name,
    }
    # 同时更新暴露控件、子图控件、模型下载元数据和模型名称枚举。
    def replace(value):
        if isinstance(value, str):
            for old, new in replacements.items():
                if value == old:
                    return new
            return value
        if isinstance(value, list):
            return [replace(v) for v in value]
        if isinstance(value, dict):
            return {k: replace(v) for k, v in value.items()}
        return value
    workflow = replace(workflow)
    for graph in iter_graphs(workflow):
        clip_nodes = []
        for node in graph["nodes"]:
            kind = node["type"]
            named = node.get("widgets_values_named", {})
            if kind == "UNETLoader":
                node["type"] = "UnetLoaderGGUF"
                node["widgets_values"] = [MANIFEST[0]["file"]]
                node["widgets_values_named"] = {"unet_name": MANIFEST[0]["file"]}
                item = MANIFEST[0]
            elif kind == "CLIPLoader":
                node["type"] = "CLIPLoader"
                node["widgets_values"] = [Path(MANIFEST[1]["file"]).name, "minimax", "default"]
                node["widgets_values_named"] = {"clip_name": Path(MANIFEST[1]["file"]).name, "type": "minimax", "device": "default"}
                clip_nodes.append(node)
                item = MANIFEST[1]
            else:
                item = None
            if item:
                node["properties"] = {"Node name for S&R": node["type"], "models": [{
                    "name": Path(item["file"]).name, "directory": item["folder"],
                    "url": f"https://huggingface.co/{item['repo']}/resolve/{item['revision']}/{item['file']}"}]}
            elif kind == "VAELoader":
                node.get("properties", {}).pop("models", None)
            elif kind == "LoraLoaderModelOnly":
                node.get("properties", {}).pop("models", None)
            if kind == "ResolutionSelector":
                set_widget(node, "megapixels", MEGAPIXELS)
            # 修改主子图暴露的控件，保留它们原本的索引和图连线。
            if "unet_name" in named and "prompt" in named:
                set_widget(node, "prompt", PROMPT)
                set_widget(node, "value_1", DURATION)
                set_widget(node, "value", True)
                set_widget(node, "value_2", TURBO_STEPS)
            if kind == "PrimitiveFloat" and "duration" in node.get("title", ""):
                set_widget(node, "value", DURATION)
            if kind == "PrimitiveBoolean":
                set_widget(node, "value", True)
            if kind == "PrimitiveInt" and node.get("inputs"):
                if any(i.get("name") == "value" and i.get("link") is not None for i in node["inputs"]):
                    set_widget(node, "value", TURBO_STEPS)
            if kind == "MarkdownNote" and node.get("title") == "Note: Model Links":
                text = ("Kaggle H3：FL2VA pruned Q4_K_M + 专用 Qwen3-VL NVFP4 AWQ。\n"
                        "GPU0：DiT/VAE；GPU1：文本编码器（需两卡可见）。双卡不是自动合并显存。\n"
                        "这是官方 H3 模板的 GGUF 适配副本；原生模板需要自行替换两个加载器。\n"
                        "先用低分辨率短片试跑，再提高分辨率和时长。图生视频请先上传自己的图片。")
                node["widgets_values"] = [text]
                node["widgets_values_named"] = {"text": text}
        if second_gpu and clip_nodes:
            # 子图采用对象形式的 links。只插入设备节点，不改 H3 算法或采样路径。
            if graph is workflow:
                raise RuntimeError("模板结构变化：加载器不在子图中")
            for clip in clip_nodes:
                outgoing = [l for l in graph["links"] if l["origin_id"] == clip["id"] and l["origin_slot"] == 0]
                node_id = max(n["id"] for n in graph["nodes"]) + 1
                link_id = max(l["id"] for l in graph["links"]) + 1
                for link in outgoing:
                    link["origin_id"] = node_id
                graph["links"].append({"id": link_id, "origin_id": clip["id"], "origin_slot": 0,
                    "target_id": node_id, "target_slot": 0, "type": "CLIP"})
                clip["outputs"][0]["links"] = [link_id]
                graph["nodes"].append({"id": node_id, "type": "SelectCLIPDevice", "pos": [-1450, 5150],
                    "size": [300, 90], "flags": {}, "order": 11, "mode": 0,
                    "inputs": [{"name": "clip", "type": "CLIP", "link": link_id}],
                    "outputs": [{"name": "CLIP", "type": "CLIP", "links": [l["id"] for l in outgoing]}],
                    "properties": {"Node name for S&R": "SelectCLIPDevice"},
                    "widgets_values": ["gpu:1"], "widgets_values_named": {"device": "gpu:1"},
                    "title": "H3 Text Encoder → GPU 1"})
                graph["state"]["lastNodeId"] = node_id
                graph["state"]["lastLinkId"] = link_id
    return workflow


print("1/6 检查实际 GPU、内存和旧服务...", flush=True)
probe = subprocess.run([sys.executable, "-c", "import json,torch; print(json.dumps({'torch':torch.__version__,'cuda':torch.version.cuda,'gpus':[{'name':torch.cuda.get_device_name(i),'vram':torch.cuda.get_device_properties(i).total_memory,'capability':torch.cuda.get_device_capability(i)} for i in range(torch.cuda.device_count())]}))"], capture_output=True, text=True)
if probe.returncode:
    raise RuntimeError("PyTorch / CUDA 检查失败：" + probe.stderr[-3000:])
hardware = json.loads(probe.stdout.strip().splitlines()[-1])
if not hardware["gpus"]:
    raise RuntimeError("未检测到 CUDA GPU。先在 Kaggle Settings > Accelerator 选择 GPU T4 x2，并重新启动会话。")
for i, gpu in enumerate(hardware["gpus"]):
    print(f"GPU {i}: {gpu['name']} / {gpu['vram']/2**30:.2f} GiB", flush=True)
second_gpu = USE_SECOND_GPU and len(hardware["gpus"]) >= 2
print("设备分配：", "DiT/VAE → GPU0；文本编码器 → GPU1" if second_gpu else "单卡 / 自动卸载模式", flush=True)
print("双卡总显存不是自动合并的显存池；本脚本显式给文本编码器指定 GPU1。", flush=True)
mem = dict(line.split(":", 1) for line in Path("/proc/meminfo").read_text().splitlines())
print(f"系统内存：{int(mem['MemTotal'].split()[0])/2**20:.1f} GiB；当前可用 {int(mem['MemAvailable'].split()[0])/2**20:.1f} GiB", flush=True)
if int(mem['MemAvailable'].split()[0]) < 24 * 2**20:
    print("⚠️ 当前可用系统内存不足 24 GiB，H3 仍可能内存不足；请停止占用资源的模型服务。", flush=True)
stop_owned("comfy_process")
with socket.socket() as sock:
    if sock.connect_ex(("127.0.0.1", PORT)) == 0:
        raise RuntimeError(f"端口 {PORT} 被其他服务占用。请停止旧 ComfyUI/Ollama 网关，或更改 PORT；没有自动杀死未知进程。")

print("2/6 更新 ComfyUI 和 H3 专用 GGUF 插件...", flush=True)
ROOT.parent.mkdir(parents=True, exist_ok=True)
clone_or_update("https://github.com/Comfy-Org/ComfyUI.git", ROOT)
plugin = ROOT / "custom_nodes/ComfyUI-GGUF"
if (plugin / ".git").is_dir():
    origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=plugin, text=True).strip()
    if "leejet/ComfyUI-GGUF" not in origin:
        backup = ROOT / "disabled_custom_nodes" / f"ComfyUI-GGUF-{int(time.time())}"
        backup.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(plugin), str(backup))
        print("已保留旧 GGUF 插件备份：", backup, flush=True)
clone_or_update("https://github.com/leejet/ComfyUI-GGUF.git", plugin)

print("3/6 安装依赖，保留 Kaggle 自带 CUDA PyTorch...", flush=True)
venv = ROOT.parent / "h3-python-env"
python = prepare_python_env(venv, sys.executable)
# 在安装其他依赖前确认 venv 仍可见 Kaggle 的 CUDA PyTorch。
torch_probe = subprocess.run([str(python), "-c",
    "import torch; assert torch.cuda.is_available(); print(torch.__version__)"],
    capture_output=True, text=True)
if torch_probe.returncode:
    raise RuntimeError("虚拟环境无法继承 Kaggle CUDA PyTorch：\n" + torch_probe.stderr[-4000:])
if torch_probe.stdout.strip().splitlines()[-1] != hardware["torch"]:
    raise RuntimeError("虚拟环境 PyTorch 版本与 Notebook 不同，请改用新的 h3-python-env 目录后重跑。")
constraints = ROOT / "h3-torch-constraints.txt"
constraints.write_text("torch==" + hardware["torch"] + "\n")
run(python, "-m", "pip", "install", "-q", "-c", constraints, "-r", ROOT / "requirements.txt", "-r", plugin / "requirements.txt")
check = subprocess.run([str(python), "-c", "import torch; assert torch.cuda.is_available(); import gguf; import av; print(torch.__version__)"], capture_output=True, text=True)
if check.returncode:
    raise RuntimeError("依赖 / CUDA 自检失败：" + check.stderr[-4000:])
print("PyTorch：", check.stdout.strip(), flush=True)

print("4/6 下载 H3 全套权重，支持断点续传和 SHA-256 校验...", flush=True)
needed = 0
for item in MANIFEST:
    target = ROOT / "models" / item["folder"] / Path(item["file"]).name
    if target.exists() and target.stat().st_size == item["size"]:
        continue
    part = target.with_name(target.name + ".part")
    partial = min(part.stat().st_size, item["size"]) if part.exists() else 0
    needed += item["size"] - partial
free = shutil.disk_usage(ROOT).free
print(f"模型合计 {sum(x['size'] for x in MANIFEST)/2**30:.2f} GiB；剩余下载约 {needed/2**30:.2f} GiB；可用磁盘 {free/2**30:.2f} GiB", flush=True)
if free < needed + 3 * 2**30:
    raise RuntimeError("磁盘不足：请为剩余下载和输出视频至少再留 3 GiB；不会自动删除现有模型。")
for item in MANIFEST:
    download_model(item)

print("5/6 保存官方 H3 模板的双卡 GGUF 适配副本，启动服务...", flush=True)
templates = json.loads(zlib.decompress(base64.b64decode(TEMPLATE_BUNDLE)))
workflow_dir = ROOT / "user/default/workflows"
workflow_dir.mkdir(parents=True, exist_ok=True)
for name, workflow in templates.items():
    workflow = patch_workflow(workflow, second_gpu)
    path = workflow_dir / f"MiniMax_H3_Kaggle_{name.upper()}.json"
    path.write_text(json.dumps(workflow, ensure_ascii=False, indent=2), encoding="utf-8")
    print("工作流：", path, flush=True)
server_log = ROOT / "comfyui-h3.log"
command = [str(python), "-u", "main.py", "--listen", "127.0.0.1", "--port", str(PORT),
           "--cuda-device", "all", "--disable-pinned-memory", "--cache-none", "--preview-method", "none"]
# 不强制 fp16/bf16，避免 T4 上 H3 超范围激活变 NaN；使用模型自身精度策略。
with server_log.open("a", encoding="utf-8") as log:
    comfy_process = subprocess.Popen(command, cwd=ROOT, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
deadline = time.monotonic() + 240
while not ready():
    if comfy_process.poll() is not None:
        raise RuntimeError("ComfyUI 启动失败：\n" + tail(server_log))
    if time.monotonic() >= deadline:
        raise TimeoutError("ComfyUI 启动超时：\n" + tail(server_log))
    time.sleep(1)
objects = request_json("/object_info", timeout=60)
required = {"MiniMaxH3ImageToVideo", "UnetLoaderGGUF", "CLIPLoader", "VAELoader", "SaveVideo", "ResolutionSelector"}
if second_gpu:
    required.add("SelectCLIPDevice")
missing = required - objects.keys()
if missing:
    raise RuntimeError("缺少节点：" + ", ".join(sorted(missing)) + "\n" + tail(server_log))
clip_spec = objects["CLIPLoader"]["input"]["required"]["type"]
clip_types = clip_spec[0] if isinstance(clip_spec[0], list) else (
    clip_spec[1].get("options") if len(clip_spec) > 1 and isinstance(clip_spec[1], dict) else None)
if clip_types is not None and "minimax" not in clip_types:
    raise RuntimeError("CLIPLoaderGGUF 没有 minimax 类型，请更新 ComfyUI 和 leejet 插件。")
if second_gpu:
    spec = objects["SelectCLIPDevice"]["input"]["required"]["device"]
    # 旧版：[选项列表, 元数据]；新版：["COMBO", {"options": 选项列表}]。
    choices = spec[0] if isinstance(spec[0], list) else (
        spec[1].get("options") if len(spec) > 1 and isinstance(spec[1], dict) else None)
    if choices is None:
        print("设备选项未在接口中展开；已检测到两张 GPU，保留工作流 gpu:1 设置。", flush=True)
    elif "gpu:1" not in choices:
        raise RuntimeError("设备节点实际选项中没有 gpu:1：" + str(choices))
    else:
        print("第二张 GPU 的设备选项检查通过：gpu:1", flush=True)
print("节点和服务就绪；尚未执行视频生成，启动成功不代表推理自检通过。", flush=True)

print("6/6 创建 / 复用 Cloudflare 临时访问链接...", flush=True)
if START_TUNNEL:
    old_tunnel = globals().get("tunnel_process")
    old_url = globals().get("url", "")
    if not (old_tunnel is not None and old_tunnel.poll() is None and
            globals().get("_h3_tunnel_target") == LOCAL_URL and
            re.fullmatch(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", str(old_url))):
        stop_owned("tunnel_process")
        binary = ROOT.parent / "cloudflared"
        if not binary.exists():
            temporary = binary.with_suffix(".download")
            source = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
            with urllib.request.urlopen(source, timeout=120) as response, temporary.open("wb") as f:
                shutil.copyfileobj(response, f)
            if temporary.read_bytes()[:4] != b"\x7fELF":
                raise RuntimeError("cloudflared 下载内容不是 Linux ELF 文件")
            temporary.replace(binary)
        binary.chmod(0o755)
        tunnel_log = ROOT / "cloudflared-h3.log"
        with tunnel_log.open("w", encoding="utf-8") as log:
            tunnel_process = subprocess.Popen([str(binary), "tunnel", "--url", LOCAL_URL, "--no-autoupdate"], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        deadline = time.monotonic() + 90
        while True:
            content = tail(tunnel_log)
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                url = match.group(0)
                _h3_tunnel_target = LOCAL_URL
                break
            if tunnel_process.poll() is not None or time.monotonic() >= deadline:
                raise RuntimeError("隧道启动失败：\n" + content)
            time.sleep(0.5)
    print("ComfyUI 访问网址：", url, flush=True)
else:
    print("ComfyUI 本地地址：", LOCAL_URL, flush=True)
print("打开 Workflows，加载 MiniMax_H3_Kaggle_T2V 或 MiniMax_H3_Kaggle_I2V。")
print("I2V 请先上传图片；首次用默认短片低分辨率测试，成功后再提高时长 / 分辨率。")
print("内置模板库中的原始 H3 模板仍使用 safetensors：请优先使用本脚本生成的适配副本，避免重复下载大模型。")
print("视频目录：", ROOT / "output/video")
print("诊断日志：", server_log)
print("临时链接可访问此 ComfyUI，会话关闭后失效；不要把链接公开发布。")


def stop_services():
    stop_owned("tunnel_process")
    stop_owned("comfy_process")
    print("本脚本的 ComfyUI 和隧道已停止。")


1/6 检查实际 GPU、内存和旧服务...
GPU 0: Tesla T4 / 14.56 GiB
GPU 1: Tesla T4 / 14.56 GiB
设备分配： DiT/VAE → GPU0；文本编码器 → GPU1
双卡总显存不是自动合并的显存池；本脚本显式给文本编码器指定 GPU1。
系统内存：31.3 GiB；当前可用 29.8 GiB
2/6 更新 ComfyUI 和 H3 专用 GGUF 插件...


Cloning into '/tmp/ComfyUI'...
Cloning into '/tmp/ComfyUI/custom_nodes/ComfyUI-GGUF'...


3/6 安装依赖，保留 Kaggle 自带 CUDA PyTorch...
pip 26.2.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)
PyTorch： 2.11.0+cu130
4/6 下载 H3 全套权重，支持断点续传和 SHA-256 校验...
模型合计 32.48 GiB；剩余下载约 32.48 GiB；可用磁盘 965.05 GiB
下载 minimax_h3_fl2va_pruned-Q4_K_M.gguf，已完成 0.00 GiB，尝试 1/3
  3.34 / 10.64 GiB
  6.95 / 10.64 GiB
校验 SHA-256： minimax_h3_fl2va_pruned-Q4_K_M.gguf.part
下载 qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors，已完成 0.00 GiB，尝试 1/3
  3.07 / 14.61 GiB
  5.02 / 14.61 GiB
  8.07 / 14.61 GiB
  10.54 / 14.61 GiB
  13.70 / 14.61 GiB
校验 SHA-256： qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors.part
下载 minimax_h3_video_vae_fp16.safetensors，已完成 0.00 GiB，尝试 1/3
  3.84 / 4.85 GiB
校验 SHA-256： minimax_h3_video_vae_fp16.safetensors.part
下载 minimax_h3_audio_vae_fp32.safetensors，已完成 0.00 GiB，尝试 1/3
校验 SHA-256： minimax_h3_audio_vae_fp32.safetensors.part
下载 minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors，已完成 0.00 GiB，尝试 1/3
校验 SHA-256： minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safete